In [ ]:
# ================================
# ADVANCED OPTUNA MULTI-OBJECTIVE TUTORIAL
# Includes:
# - Multi-objective optimization
# - Pruning (early stopping at trial level)
# - Pareto front visualization
# - Comparison vs GridSearch and RandomSearch
# ================================

import optuna
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import loguniform

# ================================
# DATA
# ================================
X, y = load_wine(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ================================
# HELPERS
# ================================
def count_parameters(model):
    total = 0
    for coef in model.coefs_:
        total += coef.size
    for intercept in model.intercepts_:
        total += intercept.size
    return total

def estimate_flops(model):
    flops = 0
    for coef in model.coefs_:
        flops += 2 * coef.size
    return flops

# ================================
# OPTUNA OBJECTIVE WITH PRUNING
# ================================
def objective(trial):

    hidden_layers = trial.suggest_categorical(
        "hidden_layer_sizes", [(50,), (100,), (50,50)]
    )
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])
    alpha = trial.suggest_float("alpha", 1e-5, 1e-1, log=True)

    model = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            alpha=alpha,
            max_iter=500,
            early_stopping=True,
            random_state=42
        )
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    f1 = f1_score(y_test, preds, average="macro")

    mlp = model.named_steps["mlpclassifier"]

    n_params = count_parameters(mlp)
    flops = estimate_flops(mlp)

    return f1, n_params, flops

# ================================
# RUN OPTUNA
# ================================
start = time.perf_counter()

study = optuna.create_study(
    directions=["maximize", "minimize", "minimize"]
)

study.optimize(objective, n_trials=30)

end = time.perf_counter()

print(f"Optuna time: {end - start:.2f} seconds")

# ================================
# PARETO FRONT
# ================================
trials = study.best_trials

df = pd.DataFrame([
    {
        "F1": t.values[0],
        "Params": t.values[1],
        "FLOPs": t.values[2]
    }
    for t in trials
])

print("\nPareto front:")
print(df)

# Plot Pareto (F1 vs Params)
plt.scatter(df["Params"], df["F1"])
plt.xlabel("Parameters")
plt.ylabel("F1 Score")
plt.title("Pareto Front (Accuracy vs Model Size)")
plt.show()

# ================================
# GRID SEARCH COMPARISON
# ================================
pipe = make_pipeline(
    StandardScaler(),
    MLPClassifier(max_iter=500, early_stopping=True, random_state=42)
)

param_grid = {
    'mlpclassifier__hidden_layer_sizes': [(50,), (100,), (50,50)],
    'mlpclassifier__activation': ['relu', 'tanh'],
    'mlpclassifier__alpha': [0.0001, 0.001]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

start = time.perf_counter()
grid = GridSearchCV(pipe, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train, y_train)
end = time.perf_counter()

print(f"Grid time: {end - start:.2f} seconds")
print("Grid best:", grid.best_score_)

# ================================
# RANDOM SEARCH COMPARISON
# ================================
param_dist = {
    'mlpclassifier__hidden_layer_sizes': [(50,), (100,), (50,50)],
    'mlpclassifier__activation': ['relu', 'tanh'],
    'mlpclassifier__alpha': loguniform(1e-5, 1e-1)
}

start = time.perf_counter()
random_search = RandomizedSearchCV(
    pipe, param_dist, n_iter=20, cv=cv, scoring='f1_macro', n_jobs=-1, random_state=42
)
random_search.fit(X_train, y_train)
end = time.perf_counter()

print(f"Random time: {end - start:.2f} seconds")
print("Random best:", random_search.best_score_)

# ================================
# TASK
# ================================
# 1. Increase trials to 50 or 100
# 2. Add deeper architectures
# 3. Compare Pareto front changes
# 4. Identify best model for:
#    - max accuracy
#    - minimum size
#    - best trade-off
